# Ollama + Ngrok Agent Setup

This notebook updates Ollama, starts a public Ngrok tunnel using `.env`, pulls the newest versioned official Llama model, and lists likely agent-ready models from the official Ollama library. It never downloads catalogue models automatically.

Run cell 5 to open the catalogue. You can browse without downloading Llama first. Use Previous, Next, or Go to navigate, Agent ready only to filter, and Refresh to reload the catalogue. When Ollama is running, use Download for a model on the current page or select and confirm an installed tag to uninstall it.

After Ollama is running, open cell 6 to chat with your downloaded models. Downloads and uninstalls from cell 5 refresh the current catalogue page and any open chat picker. Use Refresh models for changes made elsewhere.

ngrok starts automatically when configured and retries while this kernel is running. Use Offline ngrok to stop the tunnel and retries, or Online ngrok to resume. Use the current URL shown in the ngrok controls.

In Google Colab, the setup cell prompts you to upload `.env` when it is not already in `/content`. It also checks an existing mounted Drive copy at `/content/drive/MyDrive/Ollama-Ngrok/.env`:

```env
NGROK_AUTHTOKEN=your_ngrok_authtoken
```

## 1. Stop an existing ngrok tunnel (optional)

Run this on the runtime whose ngrok tunnel you want to stop.

In [ ]:
if 'ngrok_controller' in globals():
    ngrok_controller.stop()
    print(ngrok_controller.status)
else:
    from pyngrok import ngrok
    ngrok.kill()

## 2. Check the Colab GPU (optional)

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-f1d07a27-8b96-6138-4a8a-6c12b2423704)


## 3. Install and start Ollama and ngrok

In [1]:
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

def run(command, *, check=True, **kwargs):
    return subprocess.run(command, check=check, text=True, **kwargs)

def load_env(path):
    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(34)).strip(chr(39)))
    return True

def find_env_file():
    candidates = (
        Path('/content/.env'),
        Path('/content/drive/MyDrive/Ollama-Ngrok/.env'),
        Path('.env'),
    )
    return next((path for path in candidates if path.is_file()), None)

def upload_colab_env():
    try:
        from google.colab import files
    except ImportError:
        return None
    print('Upload your local .env file to configure Ngrok for this Colab runtime.')
    uploaded = files.upload()
    contents = uploaded.get('.env')
    if contents is None:
        print('No .env file was uploaded; Ngrok will not start.')
        return None
    path = Path('/content/.env')
    path.write_bytes(contents)
    return path

# Ollama's Linux release archives can require the zstd executable.
if sys.platform.startswith('linux') and shutil.which('zstd') is None:
    if shutil.which('apt-get') is None:
        raise RuntimeError('Install zstd with your system package manager, then rerun this cell.')
    apt_command = ['apt-get']
    if os.geteuid() != 0:
        if shutil.which('sudo') is None:
            raise RuntimeError('Ask your runtime administrator to install zstd, then rerun this cell.')
        apt_command = ['sudo', '-n', *apt_command]
    print('Installing zstd for Ollama archive extraction...')
    try:
        run([*apt_command, 'update', '-qq'])
        run([*apt_command, 'install', '-y', '-qq', 'zstd'])
    except subprocess.CalledProcessError as error:
        raise RuntimeError('Could not install zstd. Check the apt or sudo output above, then rerun this cell.') from error
    if shutil.which('zstd') is None:
        raise RuntimeError('zstd is still missing from PATH after installation.')

print('Installing or updating Ollama...')
ollama_install_url = 'https://' + 'ollama.com' + '/install.sh'
installer_download = subprocess.run(
    ['curl', '-fsSL', ollama_install_url], capture_output=True, text=True
)
if installer_download.returncode != 0:
    raise RuntimeError(f'Could not download the Ollama installer: {installer_download.stderr.strip()}')
installer_run = subprocess.run(
    ['bash', '-s'], input=installer_download.stdout, capture_output=True, text=True
)
if installer_run.stdout.strip():
    print(installer_run.stdout.strip())
if installer_run.returncode != 0:
    error_output = installer_run.stderr.strip() or 'The installer returned no error output.'
    raise RuntimeError(f'Ollama installer failed (exit {installer_run.returncode}): {error_output}')

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
run(['bash', '-lc', 'pkill ollama || true'], check=False)
with open('ollama.log', 'w', encoding='utf-8') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)
time.sleep(5)
run(['ollama', 'list'])

run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyngrok'])
from html import escape
from threading import Event, Lock, Thread
from urllib.error import HTTPError
from urllib.request import Request, urlopen

from pyngrok import conf, ngrok, process
from pyngrok.exception import PyngrokError


class NgrokEndpoint:
    def __init__(self):
        self.config = conf.PyngrokConfig()
        self.config.auth_token = None
        self.public_url = None
        self.status = 'Offline. Set NGROK_AUTHTOKEN in .env and rerun setup.'
        self._lock = Lock()
        self._stop = Event()
        self._stop.set()
        self._controls = None

    def configure(self, token):
        if token == self.config.auth_token:
            return
        should_start = not self._stop.is_set() or not self.config.auth_token
        if self.config.auth_token and not self.stop():
            return
        self.config.auth_token = token
        if token and should_start:
            self.start()
        else:
            self._set_status('Offline. Click Online ngrok to reconnect.' if token
                             else 'Offline. Set NGROK_AUTHTOKEN in .env and rerun setup.')

    def start(self):
        with self._lock:
            if not self._stop.is_set() or not self.config.auth_token:
                return
            self._stop = Event()
            self._set_status('Connecting…')
            try:
                self._check()
                Thread(target=self._monitor, args=(self._stop,),
                       name='ngrok-monitor', daemon=True).start()
            except (KeyboardInterrupt, RuntimeError):
                self._stop.set()
                self._set_status('Automatic reconnect stopped. Click Online ngrok to retry.')
                raise

    def stop(self):
        with self._lock:
            self._stop.set()
            try:
                ngrok.kill(pyngrok_config=self.config)
            except (PyngrokError, OSError):
                self._set_status('Could not stop the ngrok agent. Click Offline ngrok to retry.')
                return False
            self.public_url = None
            self._set_status('Offline. Automatic reconnect is off.')
            return True

    def _monitor(self, stopped):
        delay = 30
        while not stopped.wait(delay):
            with self._lock:
                if stopped.is_set():
                    return
                healthy = self._check()
            delay = 30 if healthy else min(delay * 2, 120)

    def _check(self):
        try:
            if self.public_url and process.is_process_running(self.config.ngrok_path):
                request = Request(f'{self.public_url.rstrip("/")}/api/version',
                                  headers={'ngrok-skip-browser-warning': '1'})
                try:
                    with urlopen(request, timeout=5) as response:
                        if response.headers.get_content_type() == 'application/json':
                            self._set_status('Online.')
                            return True
                    self._set_status('The endpoint responded. Ollama API health is unverified.')
                    return False
                except HTTPError as error:
                    with error:
                        code = error.headers.get('ngrok-error-code', '')
                    if code not in {'ERR_NGROK_3200', '3200'}:
                        self._set_status(f'Public API returned HTTP {error.code}. Checking again shortly.')
                        return False
                    self._set_status('ngrok reported the endpoint offline. Reconnecting…')
                    ngrok.kill(pyngrok_config=self.config)
            self.public_url = None
            tunnel = ngrok.connect(11434, pyngrok_config=self.config)
            self.public_url = tunnel.public_url
            self._set_status('Tunnel started. Checking the public API shortly.')
            return True
        except (PyngrokError, OSError, ValueError) as error:
            if 'ERR_NGROK_334' in str(error):
                self._set_status('This domain is in use. Stop the old ngrok session. Retrying automatically.')
            else:
                self._set_status('ngrok or the public API could not be reached. Retrying automatically.')
            return False

    def _set_status(self, message):
        self.status = message
        if self._controls is None:
            return
        url = self.public_url or ''
        link = ''
        if url.startswith(('https://', 'http://')):
            safe_url = escape(url)
            link = f' <a href="{safe_url}" target="_blank" rel="noopener noreferrer">{safe_url}</a>'
        self._label.value = f'<b>ngrok</b> {escape(message)}{link}'
        self._online_button.disabled = not self.config.auth_token or not self._stop.is_set()
        self._offline_button.disabled = self._stop.is_set() and not self.public_url and not message.startswith('Could not stop')

    def controls(self):
        import ipywidgets as widgets

        with self._lock:
            if self._controls is None:
                self._label = widgets.HTML()
                self._online_button = widgets.Button(description='Online ngrok', button_style='success')
                self._offline_button = widgets.Button(description='Offline ngrok', button_style='danger')
                self._online_button.on_click(lambda _: self.start())
                self._offline_button.on_click(lambda _: self.stop())
                buttons = widgets.HBox([self._online_button, self._offline_button])
                self._controls = widgets.VBox([self._label, buttons])
                self._set_status(self.status)
            return self._controls


env_file = find_env_file() or upload_colab_env()
if env_file:
    load_env(env_file)
ngrok_token = os.environ.get('NGROK_AUTHTOKEN') or os.environ.get('NGROK_TOKEN') or None
if 'ngrok_controller' not in globals():
    if ngrok_token and process.is_process_running(conf.get_default().ngrok_path):
        ngrok.kill()
    ngrok_controller = NgrokEndpoint()
ngrok_controller.configure(ngrok_token)
print(f'ngrok: {ngrok_controller.status}')
if ngrok_controller.public_url:
    print(f'Ngrok URL: {ngrok_controller.public_url}')
try:
    from IPython.display import display

    display(ngrok_controller.controls())
except ImportError:
    print('Use ngrok_controller.start() or ngrok_controller.stop() to control ngrok.')


Installing zstd for Ollama archive extraction...
Installing or updating Ollama...
ngrok: Tunnel started. Checking the public API shortly.                                             
Ngrok URL: https://manor-sniff-magical.ngrok-free.dev


## 4. Find and download the latest Llama model

In [ ]:
from functools import lru_cache
from html import unescape
from urllib.request import Request, urlopen
from urllib.parse import unquote

def fetch_text(url, timeout=30):
    request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(request, timeout=timeout) as response:
        return response.read().decode('utf-8', errors='replace')

@lru_cache(maxsize=1)
def library_models():
    html = fetch_text('https://ollama.com/library')
    slugs = re.findall(r'''href\s*=\s*["']/library/([^"']+)["']''', html)
    names = (unquote(slug) for slug in slugs)
    models = tuple(dict.fromkeys(name for name in names if re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9._-]*', name)))
    if not models:
        raise RuntimeError('No model links found in the Ollama library index. Catalogue unavailable.')
    return models

def newest_llama_model():
    candidates = []
    for name in library_models():
        match = re.fullmatch(r'llama(\d+(?:\.\d+)*)', name, flags=re.IGNORECASE)
        if match:
            version = tuple(int(part) for part in match.group(1).split('.'))
            candidates.append((version, name))
    if not candidates:
        raise RuntimeError('Could not find a versioned Llama model in the official Ollama library.')
    return max(candidates)[1]

LATEST_LLAMA_MODEL = newest_llama_model()
print(f'Pulling newest versioned Llama model: {LATEST_LLAMA_MODEL}')
run(['ollama', 'pull', LATEST_LLAMA_MODEL])
print(f'{LATEST_LLAMA_MODEL} is ready.')


Pulling newest versioned Llama model: llama4


## 5. Open the interactive model catalogue

In [3]:
import json
import re
from functools import lru_cache
from html import unescape
from urllib.error import HTTPError
from urllib.parse import unquote
from urllib.request import Request, urlopen

# Reuse the model-download cell's index when it has already run.
if 'fetch_text' not in globals() or 'library_models' not in globals():
    def fetch_text(url, timeout=30):
        request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(request, timeout=timeout) as response:
            return response.read().decode('utf-8', errors='replace')

    @lru_cache(maxsize=1)
    def library_models():
        html = fetch_text('https://ollama.com/library')
        slugs = re.findall(r'''href\s*=\s*["']/library/([^"']+)["']''', html)
        names = (unquote(slug) for slug in slugs)
        models = tuple(dict.fromkeys(name for name in names if re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9._-]*', name)))
        if not models:
            raise RuntimeError('No model links found in the Ollama library index. Catalogue unavailable.')
        return models

# This is a catalogue filter, not a guarantee that a model will work with every agent framework.
# Raise or lower MIN_CONTEXT_K / MIN_AGENT_SIGNALS to tune the result.
MODELS_PER_PAGE = 20
MIN_CONTEXT_K = 32
MIN_AGENT_SIGNALS = 2
AGENT_SIGNALS = ('tool', 'function call', 'agent', 'reasoning', 'coder', 'code', 'json')

def local_model_request(action, *, method='GET', payload=None, timeout=10):
    data = json.dumps(payload).encode('utf-8') if payload is not None else None
    request = Request(f'http://127.0.0.1:11434/api/{action}', data=data, method=method,
                      headers={'Content-Type': 'application/json'})
    try:
        return urlopen(request, timeout=timeout)
    except HTTPError as error:
        try:
            with error:
                body = json.load(error)
        except (OSError, ValueError):
            body = {}
        detail = body.get('error') if isinstance(body, dict) else None
        raise RuntimeError(detail or f'Ollama returned HTTP {error.code}.') from error

def installed_model_tags():
    with local_model_request('tags') as response:
        payload = json.load(response)
    models = payload.get('models') if isinstance(payload, dict) else None
    if not isinstance(models, list):
        raise ValueError('Ollama returned an invalid model list.')
    names = (item.get('name') or item.get('model') for item in models if isinstance(item, dict))
    return sorted({name for name in names if isinstance(name, str) and name}, key=str.lower)

def installed_models():
    try:
        return {tag.rsplit(':', 1)[0] for tag in installed_model_tags()}
    except (OSError, RuntimeError, ValueError):
        return set()

@lru_cache(maxsize=None)
def catalogue_metadata(model):
    html = fetch_text(f'https://ollama.com/library/{model}')
    description_match = re.search(r'<meta name="description" content="([^"]*)"', html)
    description = unescape(description_match.group(1)) if description_match else ''
    tags_text = re.sub(r'<[^>]+>', ' ', fetch_text(f'https://ollama.com/library/{model}/tags'))
    context_match = re.search(r'(\d+(?:\.\d+)?)\s*K\s*context', tags_text, flags=re.IGNORECASE)
    size_match = re.search(r'([0-9.]+\s?(?:GB|MB))', tags_text, flags=re.IGNORECASE)
    context_k = float(context_match.group(1)) if context_match else 0
    signals = [signal for signal in AGENT_SIGNALS if signal in f'{model} {description}'.lower()]
    size = size_match.group(1) if size_match else '---'
    return context_k, size, signals, description

def catalogue_row(model):
    try:
        context_k, size, signals, description = catalogue_metadata(model)
    except (OSError, ValueError) as error:
        return model, 'METADATA ERROR', '---', 0, [], str(error)
    missing = []
    if context_k < MIN_CONTEXT_K:
        missing.append(f'context < {MIN_CONTEXT_K}K')
    if len(signals) < MIN_AGENT_SIGNALS:
        missing.append(f'agent signals < {MIN_AGENT_SIGNALS}')
    status = 'AGENT READY' if not missing else 'NOT READY: ' + ', '.join(missing)
    return model, status, size, context_k, signals, description

def catalogue_page_data(page, *, only_agent_ready=False, refresh=False):
    if page < 1 or MODELS_PER_PAGE < 1:
        raise ValueError('Use a positive page number and page size.')
    if refresh:
        library_models.cache_clear()
        catalogue_metadata.cache_clear()
    models = sorted(library_models(), key=str.lower)
    start = (page - 1) * MODELS_PER_PAGE
    stop = start + MODELS_PER_PAGE
    errors = 0

    if only_agent_ready:
        scanned = [catalogue_row(model) for model in models]
        eligible = [row for row in scanned if row[1] == 'AGENT READY']
        errors = sum(row[1] == 'METADATA ERROR' for row in scanned)
        rows = eligible[start:stop]
        total = len(eligible)
    else:
        rows = [catalogue_row(model) for model in models[start:stop]]
        total = len(models)

    pages = (total + MODELS_PER_PAGE - 1) // MODELS_PER_PAGE
    return rows, total, pages, len(models), errors

def show_catalogue_page(page=1, *, only_agent_ready=False, refresh=False):
    if page < 1 or MODELS_PER_PAGE < 1:
        print('Use a positive page number and page size.')
        return
    rows, total, pages, index_count, errors = catalogue_page_data(
        page, only_agent_ready=only_agent_ready, refresh=refresh
    )
    if errors:
        print(f'Partial agent-ready results: metadata unavailable for {errors} model(s).')
    if not pages:
        print('No models matched the current filters.')
        return
    if page > pages:
        print(f'Choose a page from 1 to {pages}.')
        return

    installed = installed_models()
    print(f'Page {page} of {pages} | Matching models: {total} | Official index: {index_count}')
    print('Agent readiness is based on catalogue metadata. The listing covers model families.')
    for model, status, size, context_k, signals, description in rows:
        local = 'INSTALLED' if model in installed else 'NOT INSTALLED'
        print('─' * 120)
        print(f'MODEL: {model} | STATUS: {status} | LOCAL: {local}')
        print(f"SOURCE: Official Ollama | SIZE: {size} | CONTEXT: {context_k:.0f}K | SIGNALS: {', '.join(signals) or 'none'}")
        print(f"DESCRIPTION: {description or '---'}")
        print(f'URL: https://ollama.com/library/{model}')


def open_model_catalogue():
    try:
        import ipywidgets as widgets
    except ImportError:
        print('To enable catalogue controls, run %pip install ipywidgets in a separate cell and rerun this cell.')
        show_catalogue_page(page=1)
        return

    from html import escape
    from IPython.display import display

    title = widgets.HTML('<h3>Ollama Model Catalogue</h3>')
    summary = widgets.HTML()
    table = widgets.HTML()
    previous = widgets.Button(description='Previous')
    next_page = widgets.Button(description='Next')
    page_input = widgets.BoundedIntText(value=1, min=1, max=1, description='Page',
                                        layout=widgets.Layout(width='140px'))
    go = widgets.Button(description='Go', layout=widgets.Layout(width='55px'))
    ready_only = widgets.Checkbox(description='Agent ready only', indent=False)
    refresh_button = widgets.Button(description='Refresh')
    controls = [previous, next_page, page_input, go, ready_only, refresh_button]
    current_page = 1
    page_count = 0

    download_picker = widgets.Dropdown(description='On this page')
    download_tag = widgets.Text(description='Tag', placeholder='Optional, e.g. 1b')
    download = widgets.Button(description='Download', button_style='primary')
    installed_picker = widgets.Dropdown(description='Installed')
    confirm_uninstall = widgets.Checkbox(description='Confirm uninstall', indent=False)
    uninstall = widgets.Button(description='Uninstall', button_style='danger')
    local_status = widgets.Label()
    operation_status = widgets.Label()
    visible_rows = []
    local_tags = None
    managing_model = False

    def sync_action_buttons(_=None):
        unavailable = managing_model or local_tags is None
        download_picker.disabled = download_tag.disabled = unavailable or not download_picker.options
        download.disabled = unavailable or not download_picker.value
        installed_picker.disabled = unavailable or not local_tags
        confirm_uninstall.disabled = unavailable or not installed_picker.value
        uninstall.disabled = confirm_uninstall.disabled or not confirm_uninstall.value

    def reload_local_models():
        nonlocal local_tags
        selected = installed_picker.value
        try:
            local_tags = installed_model_tags()
            local_status.value = f'{len(local_tags)} installed model tag(s).'
        except (OSError, RuntimeError, ValueError):
            local_tags = None
            local_status.value = 'Local Ollama unavailable. Start Ollama and click Refresh.'
        installed_picker.options = local_tags or []
        installed_picker.value = selected if selected in installed_picker.options else next(iter(installed_picker.options), None)
        confirm_uninstall.value = False
        sync_action_buttons()
        render_model_table()

    def pull_model(name):
        with local_model_request('pull', method='POST', payload={'model': name, 'stream': True}, timeout=120) as response:
            for line in response:
                if not line.strip():
                    continue
                progress = json.loads(line)
                if not isinstance(progress, dict):
                    raise ValueError('Ollama returned invalid download progress.')
                if progress.get('error'):
                    raise RuntimeError(progress['error'])
                status = progress.get('status', 'Downloading')
                if status == 'success':
                    return
                total, completed = progress.get('total'), progress.get('completed', 0)
                if isinstance(total, int) and total > 0 and isinstance(completed, int):
                    status = f'{status} ({min(100, max(0, completed * 100 // total))}% of current layer)'
                operation_status.value = f'{name} · {status}'
        raise RuntimeError('The download ended before Ollama reported success.')

    def change_model(action, name):
        nonlocal managing_model
        if managing_model or local_tags is None:
            return
        managing_model = True
        for control in controls:
            control.disabled = True
        sync_action_buttons()
        operation_status.value = f'{action} requested for {name}…'
        changed = False
        try:
            if action == 'Download':
                pull_model(name)
                operation_status.value = f'Downloaded {name}.'
            else:
                with local_model_request('delete', method='DELETE', payload={'model': name}):
                    pass
                operation_status.value = f'Uninstalled {name}.'
        except (OSError, RuntimeError, ValueError) as error:
            operation_status.value = f'{action} failed for {name}. {error}'
        except KeyboardInterrupt:
            operation_status.value = f'Stopped waiting for {action.lower()} of {name}.'
        else:
            changed = True
        finally:
            managing_model = False
            reload_local_models()
            sync_navigation()
        refresh_chat_models = globals().get('_refresh_downloaded_model_chat')
        if changed and callable(refresh_chat_models):
            refresh_chat_models(tags=local_tags)

    def download_selected(_):
        family = download_picker.value
        if family not in download_picker.options:
            return
        tag = download_tag.value.strip()
        if tag and not re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9._-]*', tag):
            operation_status.value = 'Enter a tag such as 1b or leave the field empty for the default tag.'
            return
        name = f'{family}:{tag}' if tag else family
        change_model('Download', name)

    def uninstall_selected(_):
        selected = installed_picker.value
        if local_tags and selected in local_tags and confirm_uninstall.value:
            confirm_uninstall.value = False
            change_model('Uninstall', selected)

    def render_model_table():
        if not visible_rows:
            table.value = ''
            return
        installed = {tag.rsplit(':', 1)[0] for tag in local_tags or []}
        columns = ('Model', 'Local status', 'Agent readiness', 'Size', 'Context', 'Signals', 'Description')
        header = ''.join(f'<th scope="col" style="text-align:left;padding:6px">{column}</th>' for column in columns)
        body = []
        for model, status, size, context_k, signals, description in visible_rows:
            name = escape(model)
            link = f'<a href="https://ollama.com/library/{name}" target="_blank" rel="noopener noreferrer">{name}</a>'
            values = (
                'Unknown' if local_tags is None else ('Installed' if model in installed else 'Not installed'),
                status, size, f'{context_k:g}K' if context_k else '—',
                ', '.join(signals) or '—', description or '—',
            )
            cells = ''.join(f'<td style="padding:6px">{escape(value)}</td>' for value in values)
            body.append(f'<tr><th scope="row" style="text-align:left;padding:6px">{link}</th>{cells}</tr>')
        table.value = (
            '<div style="overflow-x:auto"><table style="width:100%;border-collapse:collapse">'
            '<caption>Agent readiness is estimated from catalogue metadata.</caption>'
            f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table></div>'
        )

    def sync_navigation():
        for control in controls:
            control.disabled = False
        previous.disabled = page_count == 0 or current_page <= 1
        next_page.disabled = current_page >= page_count
        page_input.disabled = go.disabled = page_count <= 1

    def render(page=1, *, refresh=False):
        nonlocal current_page, page_count, visible_rows
        for control in controls:
            control.disabled = True
        summary.value = ('Checking agent readiness across the full index. This can take a while.'
                         if ready_only.value else 'Loading catalogue page…')
        table.value = ''
        visible_rows = []
        try:
            rows, total, pages, index_count, errors = catalogue_page_data(
                page, only_agent_ready=ready_only.value, refresh=refresh
            )
            visible_rows = rows
            current_page, page_count = page, pages
            page_input.max = max(1, pages)
            page_input.value = page
            summary.value = (
                f'<b>Page {page} of {pages}</b> · {total} matching models · {index_count} in the official index'
                if pages else f'<b>No models match the current filter.</b> · {index_count} in the official index'
            )
            if errors:
                summary.value += f'<br>Partial results: metadata unavailable for {errors} model(s).'
        except (OSError, ValueError, RuntimeError) as error:
            page_count = 0
            summary.value = f'Could not load the catalogue. Use Refresh to retry. {escape(str(error))}'
        finally:
            selected = download_picker.value
            download_picker.options = [row[0] for row in visible_rows]
            download_picker.value = selected if selected in download_picker.options else next(iter(download_picker.options), None)
            sync_navigation()
            reload_local_models()

    def reset_uninstall_confirmation(_):
        confirm_uninstall.value = False
        sync_action_buttons()

    previous.on_click(lambda _: render(current_page - 1))
    next_page.on_click(lambda _: render(current_page + 1))
    go.on_click(lambda _: render(page_input.value))
    ready_only.observe(lambda _: render(), names='value')
    refresh_button.on_click(lambda _: render(refresh=True))
    download_picker.observe(lambda _: setattr(download_tag, 'value', ''), names='value')
    installed_picker.observe(reset_uninstall_confirmation, names='value')
    confirm_uninstall.observe(sync_action_buttons, names='value')
    download.on_click(download_selected)
    uninstall.on_click(uninstall_selected)
    layout = widgets.Layout(flex_flow='row wrap', align_items='center')
    toolbar = widgets.HBox(controls, layout=layout)
    download_controls = widgets.HBox([download_picker, download_tag, download], layout=layout)
    uninstall_controls = widgets.HBox([installed_picker, confirm_uninstall, uninstall], layout=layout)
    sections = [title]
    if 'ngrok_controller' in globals():
        sections.append(ngrok_controller.controls())
    display(widgets.VBox([*sections, toolbar, download_controls, uninstall_controls,
                          local_status, operation_status, summary, table]))
    render()

open_model_catalogue()

## 6. Chat with downloaded models

In [ ]:
def open_downloaded_model_chat():
    import json
    from html import escape
    from urllib.error import HTTPError
    from urllib.request import Request, urlopen

    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print('Run %pip install ipywidgets in a separate cell, then rerun this cell.')
        return

    model = widgets.Dropdown(description='Model', layout=widgets.Layout(width='100%'))
    refresh = widgets.Button(description='Refresh models')
    clear = widgets.Button(description='Clear chat')
    prompt = widgets.Textarea(placeholder='Type a message…', layout=widgets.Layout(width='100%'))
    send = widgets.Button(description='Send', button_style='primary')
    status = widgets.Label()
    transcript = widgets.HTML()
    histories = {}

    def request_api(path, payload=None, *, timeout=10):
        data = json.dumps(payload).encode('utf-8') if payload is not None else None
        request = Request(f'http://127.0.0.1:11434/api/{path}', data=data,
                          headers={'Content-Type': 'application/json'})
        try:
            with urlopen(request, timeout=timeout) as response:
                result = json.load(response)
        except HTTPError as error:
            try:
                with error:
                    body = json.load(error)
            except (OSError, ValueError):
                body = {}
            detail = body.get('error') if isinstance(body, dict) else None
            raise RuntimeError(detail or f'Ollama returned HTTP {error.code}.') from error
        if not isinstance(result, dict):
            raise ValueError('Ollama returned an invalid response.')
        return result

    def render():
        messages = histories.get(model.value, [])
        transcript.value = ''.join(
            f'<div style="margin:0.5em 0"><b>{role_labels[message["role"]]}</b>'
            f'<div style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(message["content"])}</div></div>'
            for message in messages
        )
        send.disabled = prompt.disabled = not model.value
        clear.disabled = not messages

    role_labels = {'user': 'You', 'assistant': 'Model'}

    def refresh_models(_=None, *, tags=None):
        selected = model.value
        try:
            if tags is None:
                models = request_api('tags').get('models', [])
                if not isinstance(models, list):
                    raise ValueError('Ollama returned an invalid model list.')
                names = (item.get('name') or item.get('model') for item in models if isinstance(item, dict))
                tags = sorted({name for name in names if isinstance(name, str) and name}, key=str.lower)
            model.options = tags
            model.value = selected if selected in model.options else next(iter(model.options), None)
            status.value = ('Select a model to chat.' if model.options
                            else 'No downloaded models found. Download a model and click Refresh models.')
        except (OSError, RuntimeError, ValueError) as error:
            model.options = []
            model.value = None
            status.value = f'Could not list models. Make sure Ollama is running, then refresh. {error}'
        render()

    def send_message(_):
        selected = model.value
        content = prompt.value.strip()
        if not selected or not content:
            status.value = 'Select a model and enter a message.'
            return

        messages = [*histories.get(selected, []), {'role': 'user', 'content': content}]
        inputs = (model, refresh, clear, prompt, send)
        for control in inputs:
            control.disabled = True
        status.value = f'Waiting for {selected}…'
        try:
            result = request_api('chat', {'model': selected, 'messages': messages, 'stream': False}, timeout=300)
            reply = result.get('message')
            answer = reply.get('content') if isinstance(reply, dict) else None
            if not isinstance(answer, str) or not answer.strip():
                raise ValueError('The model returned no text. Try a model that supports chat.')
            histories[selected] = [*messages, {'role': 'assistant', 'content': answer}]
            prompt.value = ''
            status.value = ''
        except (OSError, RuntimeError, ValueError) as error:
            status.value = f'Chat failed for {selected}. {error}'
        except KeyboardInterrupt:
            status.value = 'Chat interrupted.'
        finally:
            for control in inputs:
                control.disabled = False
            render()

    def clear_chat(_):
        histories.pop(model.value, None)
        status.value = 'Cleared the selected model’s chat.'
        render()

    model.observe(lambda _: render(), names='value')
    refresh.on_click(refresh_models)
    clear.on_click(clear_chat)
    send.on_click(send_message)
    display(widgets.VBox([model, widgets.HBox([refresh, clear]), transcript, prompt, send, status]))
    refresh_models()
    return refresh_models


_refresh_downloaded_model_chat = open_downloaded_model_chat()